# 11 · Model solution: insurance targeting

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lukasquaderer-lgtm/Linear-regression/blob/master/notebooks/solutions/11_project_insurance_targeting_solution.ipynb)

One reasonable way to do the project. Your numbers will differ if you made different choices; what matters is the
process: an honest split, a metric that matches the decision, baselines, and a clear recommendation.

In [1]:
# Setup: run this cell first (Shift + Enter). Works in Jupyter, VS Code and Google Colab.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from pathlib import Path
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)   # hide library deprecation notices

# Use the local data folder if it exists, otherwise read the CSVs straight from GitHub (e.g. in Colab)
DATA = next((p for p in ["../data/", "../../data/"] if Path(p).exists()), "https://raw.githubusercontent.com/lukasquaderer-lgtm/Linear-regression/master/data/")
pd.set_option("display.precision", 4)
plt.rcParams["figure.figsize"] = (8, 4)

# Answer checker: after an exercise, check("01.1") tells you whether your answer is right
import sys
if Path("../checks.py").exists():
    sys.path.insert(0, "..")  # solution notebooks live one folder down
try:
    from checks import check, check_all
except ImportError:  # in Colab: fetch the checker from GitHub
    import urllib.request
    urllib.request.urlretrieve("https://raw.githubusercontent.com/lukasquaderer-lgtm/Linear-regression/master/notebooks/checks.py", "checks.py")
    from checks import check, check_all
print("Ready. Data folder:", DATA)

Ready. Data folder: ../../data/


In [2]:
# Load a real dataset from the ISLP package (Introduction to Statistical Learning with Python)
import subprocess, sys
def load_islp(name):
    try:
        import ISLP
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "ISLP"], check=True)
        import ISLP
    return pd.read_csv(Path(ISLP.__file__).parent / "data" / f"{name}.csv")

In [3]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, GridSearchCV
from sklearn.metrics import make_scorer, roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
import warnings; warnings.filterwarnings("ignore")

car = load_islp("Caravan")
X = car.drop(columns="Purchase"); y = (car["Purchase"] == "Yes").astype(int)
base = y.mean()
print(f"1) Base rate {base:.2%}: a random 10% mailing (582 letters) reaches about {0.10 * len(y) * base:.0f} buyers")

def precision_at_top(y_true, scores, frac=0.10):
    k = int(np.ceil(frac * len(scores)))
    return np.asarray(y_true)[np.argsort(np.asarray(scores))[::-1][:k]].mean()
top10 = make_scorer(precision_at_top, response_method="predict_proba")

1) Base rate 5.98%: a random 10% mailing (582 letters) reaches about 35 buyers


In [4]:
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=0, stratify=y)     # 2) test set locked away
cv = StratifiedKFold(5, shuffle=True, random_state=0)

# 3) baselines
rng = np.random.default_rng(0)
rand = np.mean([precision_at_top(y_tr.iloc[te], rng.random(len(te))) for _, te in cv.split(X_tr, y_tr)])
rule = np.mean([precision_at_top(y_tr.iloc[te], X_tr["PPERSAUT"].iloc[te] + 1e-6 * rng.random(len(te))) for _, te in cv.split(X_tr, y_tr)])
print(f"3) precision in top 10% | random: {rand:.3f} | rule 'rank by PPERSAUT': {rule:.3f}")

# 4) model comparison on the same folds
models = {
    "Logistic (L2)": make_pipeline(StandardScaler(), LogisticRegression(C=0.05, max_iter=2000)),
    "Logistic (L1)": make_pipeline(StandardScaler(), LogisticRegression(C=0.05, penalty="l1", solver="liblinear")),
    "Random forest": RandomForestClassifier(n_estimators=400, min_samples_leaf=20, random_state=0, n_jobs=-1),
    "Gradient boosting": HistGradientBoostingClassifier(learning_rate=0.05, max_depth=3, random_state=0),
}
rows = []
for name, m in models.items():
    r = cross_validate(m, X_tr, y_tr, cv=cv, scoring={"top10": top10, "auc": "roc_auc"})
    rows.append([name, r["test_top10"].mean(), r["test_top10"].std(), r["test_auc"].mean()])
pd.DataFrame(rows, columns=["model", "precision@10%", "± std", "AUC"]).set_index("model").round(3)

3) precision in top 10% | random: 0.057 | rule 'rank by PPERSAUT': 0.114


,precision@10%,± std,AUC
model,,,
Logistic (L2),0.180,0.038,0.738
Logistic (L1),0.195,0.032,0.740
Random forest,0.223,0.027,0.739
Gradient boosting,0.202,0.033,0.749


All models roughly double or triple the random hit rate, and the fold-to-fold standard deviation is large: with only ~260 buyers in training, differences of 0.01–0.02 are noise. A simple regularised logistic regression is competitive.

In [5]:
# 5) tune the logistic model's penalty
gs = GridSearchCV(make_pipeline(StandardScaler(), LogisticRegression(penalty="l1", solver="liblinear")),
                  {"logisticregression__C": [0.005, 0.01, 0.02, 0.05, 0.1, 0.5]}, cv=cv, scoring=top10).fit(X_tr, y_tr)
print("5) best C:", gs.best_params_, "| CV precision@10%:", round(gs.best_score_, 3))
final = gs.best_estimator_

# 6) the one and only test evaluation
s_te = final.predict_proba(X_te)[:, 1]
p10 = precision_at_top(y_te, s_te)
print(f"6) test precision@10% = {p10:.3f} | lift = {p10 / y_te.mean():.1f}x | buyers per 100 letters: {100 * p10:.0f} vs {100 * y_te.mean():.0f} at random | AUC {roc_auc_score(y_te, s_te):.3f}")

5) best C: {'logisticregression__C': 0.05} | CV precision@10%: 0.195
6) test precision@10% = 0.192 | lift = 3.2x | buyers per 100 letters: 19 vs 6 at random | AUC 0.744


In [6]:
# 7) what drives the ranking? (permutation importance on the held-out test set)
pi = permutation_importance(final, X_te, y_te, scoring=top10, n_repeats=20, random_state=0)
imp = pd.Series(pi.importances_mean, index=X.columns).sort_values(ascending=False).head(10)
print(imp.round(4))
coef = pd.Series(final[-1].coef_[0], index=X.columns)
print("\nNon-zero L1 coefficients:", int((coef != 0).sum()), "of", len(coef)); print(coef[coef != 0].sort_values().round(3))

PPERSAUT    0.0719
PBRAND      0.0301
APLEZIER    0.0243
MKOOPKLA    0.0185
MOPLHOOG    0.0158
MOPLLAAG    0.0154
PWAPART     0.0103
MBERMIDD    0.0096
MRELGE      0.0092
MINKM30     0.0082
dtype: float64

Non-zero L1 coefficients: 26 of 85
MOPLLAAG   -0.065
MGODGE     -0.060
MRELSA     -0.042
MINKM30    -0.041
MBERBOER   -0.038
AWALAND    -0.035
MHHUUR     -0.027
ABROM      -0.017
MGEMLEEF    0.001
AZEILPL     0.009
AINBOED     0.013
AFIETS      0.015
ALEVEN      0.020
MBERMIDD    0.023
PGEZONG     0.033
MAUT1       0.042
PWAOREG     0.059
MRELGE      0.072
ABYSTAND    0.074
MKOOPKLA    0.080
MOPLHOOG    0.087
MINK7512    0.098
PBRAND      0.102
APLEZIER    0.122
PWAPART     0.158
PPERSAUT    0.515
dtype: float64


### 8) Recommendation (example)
Mail the 10% of customers ranked highest by the model: on unseen data, roughly **3× as many buyers per letter as a
random mailing** (about 19 buyers per 100 letters instead of 6). Buyers are concentrated among customers who already hold car insurance with us (`PPERSAUT`) and
fire insurance (`PBRAND`), and in higher purchasing-power areas, so cross-selling to existing car-policy holders is the
core of the campaign. The random forest scored slightly higher in cross-validation, but within the fold-to-fold noise,
so I chose the lasso-penalised logistic model: it keeps only about a third of the features and is easy to explain and audit. Main risk: only about 350 buyers exist in the data, so the hit rate could vary by several percentage points from
campaign to campaign; run the first wave as a test against a random control group.